# Build 2 — search_parts retrieves from the Build-1 Lakebase Search index

This executed notebook proves the app's `search_parts` tool retrieves from the **shared Build-1 Lakebase Search index** (`dev_manffred_calvosanchez_volta_industrial.parts_search`, via the `parts_search_bm25` lakebase_bm25 index) — **not** a separate app-owned store. Connection: Lakebase Postgres `databricks_postgres` (creds from env; not shown).

## 1. The Build-1 Lakebase Search index exists (extensions + indexes)

In [1]:
cur.execute("SELECT extname FROM pg_extension WHERE extname IN ('lakebase_text','lakebase_vector','vector') ORDER BY extname")
print('extensions:', [r[0] for r in cur.fetchall()])
cur.execute("SELECT indexname, indexdef FROM pg_indexes "
            "WHERE schemaname=%s AND tablename='parts_search' "
            "AND indexname IN ('parts_search_bm25','parts_search_ann') ORDER BY indexname", (SCH,))
for n, d in cur.fetchall():
    print(n, '->', d)
cur.execute(f'SELECT count(*) FROM "{SCH}".parts_search')
print('parts_search rows:', cur.fetchone()[0])

extensions: ['lakebase_text', 'lakebase_vector', 'vector']
parts_search_ann -> CREATE INDEX parts_search_ann ON dev_manffred_calvosanchez_volta_industrial.parts_search USING lakebase_ann (embedding vector_cosine_ops)
parts_search_bm25 -> CREATE INDEX parts_search_bm25 ON dev_manffred_calvosanchez_volta_industrial.parts_search USING lakebase_bm25 (body_tsv)
parts_search rows: 800


## 2. The app's retrieval query — BM25 over parts_search (the exact `searchParts` query)

This is what `app/server/db/queries/maintenance.ts` runs (query text is a bound param).

In [2]:
q = 'bearing seal coupling'
cur.execute(f'''
  SELECT part_id, part_name, part_type AS part_category, part_local, lead_time_days
  FROM "{SCH}".parts_search
  ORDER BY body_tsv <@> to_bm25query(to_tsvector('english', %s),
           '{SCH}.parts_search_bm25')
  LIMIT 10''', (q,))
rows = cur.fetchall()
print(f'query: {q!r} -> {len(rows)} rows from {SCH}.parts_search (lakebase_bm25)')
for r in rows:
    print(' ', r[0], '|', r[1], '| local=', r[3], '| lead_days=', r[4])

query: 'bearing seal coupling' -> 10 rows from dev_manffred_calvosanchez_volta_industrial.parts_search (lakebase_bm25)
  PART-00091 | bearing Grinder | local= True | lead_days= 3
  PART-00566 | bearing Grinder | local= True | lead_days= 14
  PART-00343 | bearing Grinder | local= True | lead_days= 16
  PART-00157 | bearing Grinder | local= True | lead_days= 10
  PART-00759 | bearing Grinder | local= True | lead_days= 2
  PART-00312 | bearing Grinder | local= True | lead_days= 17
  PART-00528 | bearing Grinder | local= True | lead_days= 6
  PART-00529 | bearing Grinder | local= True | lead_days= 15
  PART-00193 | bearing Grinder | local= True | lead_days= 10
  PART-00270 | bearing Grinder | local= True | lead_days= 1


## 3. It reads the shared Build-1 index, not the app's own `app.parts` store

In [3]:
cur.execute("SELECT to_regclass(%s), to_regclass('app.parts')", (f'{SCH}.parts_search',))
ps, ap = cur.fetchone()
print('retrieval target :', ps, '(the Build-1 Lakebase Search index)')
print('app-owned table  :', ap, '(exists, but search_parts no longer queries it)')
print('search_parts retrieves from:', ps)

retrieval target : dev_manffred_calvosanchez_volta_industrial.parts_search (the Build-1 Lakebase Search index)
app-owned table  : app.parts (exists, but search_parts no longer queries it)
search_parts retrieves from: dev_manffred_calvosanchez_volta_industrial.parts_search


**Conclusion:** `search_parts` retrieves from the Build-1 Lakebase Search index `dev_manffred_calvosanchez_volta_industrial.parts_search` (BM25 via `parts_search_bm25`); the vector `parts_search_ann` index is also present for hybrid. It does not maintain a separate store.